Tài lên Kafka

In [4]:
!pip install minio confluent_kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 85.6 MB/s eta 0:00:00


In [5]:
import requests
import json
import time
from confluent_kafka import Producer
from datetime import datetime

# --- 1. CẤU HÌNH KAFKA ---
KAFKA_HOST = '18.219.196.163:9092'  # Nếu chạy trên EC2
TOPIC_NAME = 'weather_multi_city'

# --- 2. CẤU HÌNH 4 THÀNH PHỐ ---
LOCATIONS = [
    {"city": "Ha Noi",    "region": "North",   "sensor_id": "SENSOR_HN_01", "lat": 21.0285, "lon": 105.8542},
    {"city": "Hai Phong", "region": "North",   "sensor_id": "SENSOR_HP_01", "lat": 20.8449, "lon": 106.6881},
    {"city": "Da Nang",   "region": "Central", "sensor_id": "SENSOR_DN_01", "lat": 16.0471, "lon": 108.2068},
    {"city": "Can Tho",   "region": "South",   "sensor_id": "SENSOR_CT_01", "lat": 10.0452, "lon": 105.7469}
]

# --- 3. URL API & DANH SÁCH BIẾN ---
API_URL = "https://api.open-meteo.com/v1/forecast"
# Các biến lấy từ API (13 biến)
CURRENT_VARS = ",".join([
    "temperature_2m", "relative_humidity_2m", "apparent_temperature",
    "weather_code", "pressure_msl", "rain", "wind_speed_10m",
    "wind_direction_10m", "wind_gusts_10m", "shortwave_radiation",
    "visibility", "cloud_cover", "is_day"
])

def main():
    conf = {'bootstrap.servers': KAFKA_HOST, 'client.id': 'producer-17-cols'}
    producer = Producer(conf)

    SLEEP_SECONDS = 3600  # 1 tiếng lấy 1 lần

    print(f"🚀 BẮT ĐẦU JOB: Lấy đủ 17 cột dữ liệu cho 4 thành phố")
    print("-" * 50)

    while True:
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n[{now}] Đang quét...")

        for loc in LOCATIONS:
            params = {
                "latitude": loc["lat"], "longitude": loc["lon"],
                "current": CURRENT_VARS, "timezone": "Asia/Bangkok"
            }

            try:
                res = requests.get(API_URL, params=params, timeout=15)
                if res.status_code == 200:
                    data = res.json()
                    curr = data.get('current', {})

                    # --- ĐÓNG GÓI 17 CỘT DỮ LIỆU ---
                    message = {
                        # Nhóm 1: Định danh
                        "sensor_id": loc["sensor_id"],           # Cột 1
                        "city": loc["city"],                     # Cột 2
                        "region": loc["region"],                 # Cột 3

                        # Nhóm 2: Thời gian
                        "timestamp": curr.get("time"),           # Cột 4

                        # Nhóm 3: Chỉ số môi trường
                        "temperature": curr.get("temperature_2m"),       # Cột 5
                        "humidity": curr.get("relative_humidity_2m"),    # Cột 6
                        "apparent_temperature": curr.get("apparent_temperature"), # Cột 7
                        "weather_code": curr.get("weather_code"),        # Cột 8
                        "pressure": curr.get("pressure_msl"),            # Cột 9
                        "rain": curr.get("rain"),                        # Cột 10
                        "wind_speed": curr.get("wind_speed_10m"),        # Cột 11
                        "wind_direction": curr.get("wind_direction_10m"),# Cột 12
                        "wind_gusts": curr.get("wind_gusts_10m"),        # Cột 13
                        "solar_radiation": curr.get("shortwave_radiation"), # Cột 14
                        "visibility": curr.get("visibility"),            # Cột 15
                        "cloud_cover": curr.get("cloud_cover"),          # Cột 16
                        "is_day": curr.get("is_day")                     # Cột 17
                    }

                    # Gửi Kafka
                    producer.produce(TOPIC_NAME, value=json.dumps(message).encode('utf-8'))
                    producer.poll(0)
                    print(f"  ✅ {loc['city']}: Đã gửi đủ 17 trường dữ liệu.")

            except Exception as e:
                print(f"  ❌ Lỗi {loc['city']}: {e}")

            time.sleep(1)

        producer.flush()
        print(f"😴 Ngủ 1 tiếng...")
        time.sleep(SLEEP_SECONDS)

if __name__ == "__main__":
    main()

🚀 BẮT ĐẦU JOB: Lấy đủ 17 cột dữ liệu cho 4 thành phố
--------------------------------------------------

[2026-01-31 12:53:06] Đang quét...
  ✅ Ha Noi: Đã gửi đủ 17 trường dữ liệu.
  ✅ Hai Phong: Đã gửi đủ 17 trường dữ liệu.
  ✅ Da Nang: Đã gửi đủ 17 trường dữ liệu.
  ✅ Can Tho: Đã gửi đủ 17 trường dữ liệu.
😴 Ngủ 1 tiếng...


KeyboardInterrupt: 